In [1]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception as e:
    print("Not running on Colab or Drive already mounted:", e)

!pip -q install lightgbm joblib

Mounted at /content/drive


In [2]:

import os
import re
import json
import time
import warnings
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import joblib

from scipy.sparse import hstack, csr_matrix, save_npz, load_npz

from sklearn.base import clone
from sklearn.pipeline import FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.tree import DecisionTreeClassifier

from lightgbm import LGBMClassifier

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

BASE_DIR = "/content/drive/MyDrive/CS114/TeamModel"
DATA_DIR = os.path.join(BASE_DIR, "data")

TRAIN_PATH = os.path.join(DATA_DIR, "train_clean_tokenizedEmoji.csv")
VAL_PATH   = os.path.join(DATA_DIR, "val_clean_tokenizedEmoji.csv")
TEST_PATH  = os.path.join(DATA_DIR, "test_clean_tokenizedEmoji.csv")

EXPERIMENT_NAME = "tfidf_lexicon_only_all_default_models_decision_tree_v1"

EXP_DIR = os.path.join(BASE_DIR, "experiments", EXPERIMENT_NAME)
CHECKPOINT_DIR = os.path.join(EXP_DIR, "checkpoints")
REPORT_DIR = os.path.join(EXP_DIR, "reports")
FIGURE_DIR = os.path.join(EXP_DIR, "figures")

for d in [EXP_DIR, CHECKPOINT_DIR, REPORT_DIR, FIGURE_DIR]:
    os.makedirs(d, exist_ok=True)

WORD_NGRAM_RANGE = (1, 3)
CHAR_NGRAM_RANGE = (3, 5)
CHAR_ANALYZER = "char_wb"

WORD_MAX_FEATURES = 30000
CHAR_MAX_FEATURES = 30000

WORD_MIN_DF = 3
CHAR_MIN_DF = 3

FEATURE_K = 35000

DROP_CROSS_SPLIT_DUPLICATES = True
FORCE_REBUILD_FEATURES = False
FORCE_RETRAIN_MODELS = False


print("Experiment directory:", EXP_DIR)

Experiment directory: /content/drive/MyDrive/CS114/TeamModel/experiments/tfidf_lexicon_only_all_default_models_decision_tree_v1


In [3]:
def load_split(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    required_cols = {"text", "status"}
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing columns in {path}: {missing}")

    df = df.dropna(subset=["text", "status"]).copy()
    df["text"] = df["text"].astype(str)
    df["status"] = df["status"].astype(str).str.strip().str.lower()
    return df

train_df = load_split(TRAIN_PATH)
val_df = load_split(VAL_PATH)
test_df = load_split(TEST_PATH)

print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

print("\nTrain label distribution:")
print(train_df["status"].value_counts())

print("\nValidation label distribution:")
print(val_df["status"].value_counts())

print("\nTest label distribution:")
print(test_df["status"].value_counts())

Train: (34601, 2)
Val  : (4943, 2)
Test : (9887, 2)

Train label distribution:
status
normal        12706
depression    10154
suicidal       7840
anxiety        3901
Name: count, dtype: int64

Validation label distribution:
status
normal        1815
depression    1451
suicidal      1120
anxiety        557
Name: count, dtype: int64

Test label distribution:
status
normal        3630
depression    2902
suicidal      2240
anxiety       1115
Name: count, dtype: int64


In [4]:
def normalize_text_key(text: str) -> str:
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

if DROP_CROSS_SPLIT_DUPLICATES:
    train_df["text_key"] = train_df["text"].apply(normalize_text_key)
    val_df["text_key"] = val_df["text"].apply(normalize_text_key)
    test_df["text_key"] = test_df["text"].apply(normalize_text_key)

    train_keys = set(train_df["text_key"])
    val_before = len(val_df)
    test_before = len(test_df)

    val_df = val_df[~val_df["text_key"].isin(train_keys)].copy()

    train_val_keys = set(train_df["text_key"]) | set(val_df["text_key"])
    test_df = test_df[~test_df["text_key"].isin(train_val_keys)].copy()

    print(f"Removed from val : {val_before - len(val_df)} duplicates")
    print(f"Removed from test: {test_before - len(test_df)} duplicates")

    train_df = train_df.drop(columns=["text_key"])
    val_df = val_df.drop(columns=["text_key"])
    test_df = test_df.drop(columns=["text_key"])

print("Final split sizes:")
print("Train:", train_df.shape)
print("Val  :", val_df.shape)
print("Test :", test_df.shape)

Removed from val : 4 duplicates
Removed from test: 12 duplicates
Final split sizes:
Train: (34601, 2)
Val  : (4939, 2)
Test : (9875, 2)


In [5]:
DEPRESSION_KEYWORDS = [
    "depression", "depressed", "depressing", "sad", "sadness",
    "hopeless", "hopelessness", "worthless", "empty", "numb",
    "lonely", "alone", "miserable", "unmotivated", "no motivation",
    "tired of everything", "can't sleep", "cannot sleep", "insomnia",
    "crying", "cry", "lost interest", "nothing matters"
]

SUICIDE_KEYWORDS = [
    "suicide", "suicidal", "kill myself", "kms", "kys",
    "end my life", "want to die", "wanna die", "wish i was dead",
    "don't want to live", "do not want to live", "tired of living",
    "overdose", "hang myself", "cut myself", "self harm",
    "self-harm", "take my life", "ending it all", "goodbye everyone"
]

def normalize_for_lexicon(text: str) -> str:
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def count_keywords(text: str, keywords) -> int:
    text = normalize_for_lexicon(text)
    count = 0
    for kw in keywords:
        kw = kw.lower().strip()
        pattern = r"\b" + re.escape(kw) + r"\b"
        count += len(re.findall(pattern, text))
    return count

def reduce_long_repetitions(text: str) -> str:
    return re.sub(r"(.)\1{4,}", r"\1\1", str(text))

def extract_lexicon_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    raw_text = out["text"].astype(str)

    out["cleaned_text"] = (
        raw_text
        .str.lower()
        .str.strip()
        .apply(reduce_long_repetitions)
    )

    out["depression_keyword_count"] = raw_text.apply(
        lambda x: count_keywords(x, DEPRESSION_KEYWORDS)
    )

    out["suicide_keyword_count"] = raw_text.apply(
        lambda x: count_keywords(x, SUICIDE_KEYWORDS)
    )

    return out

train_enriched = extract_lexicon_features(train_df)
val_enriched = extract_lexicon_features(val_df)
test_enriched = extract_lexicon_features(test_df)

LEXICON_FEATURES = [
    "depression_keyword_count",
    "suicide_keyword_count"
]

y_train = train_enriched["status"].values
y_val = val_enriched["status"].values
y_test = test_enriched["status"].values

class_list = sorted(train_enriched["status"].unique().tolist())
print("Classes:", class_list)

print("\\nFeature preview:")
display(train_enriched[["text", "cleaned_text"] + LEXICON_FEATURES].head())

Classes: ['anxiety', 'depression', 'normal', 'suicidal']
\nFeature preview:


,text,cleaned_text,depression_keyword_count,suicide_keyword_count
0,is anime bad ? is anime bad ? i think it's bad...,is anime bad ? is anime bad ? i think it's bad...,0,0
1,horror wench me too i feel like i ve been on t...,horror wench me too i feel like i ve been on t...,0,0
2,context: my father-in-law is/was the ultimate ...,context: my father-in-law is/was the ultimate ...,0,0
3,been trying the whole online dating thing have...,been trying the whole online dating thing have...,1,0
4,"selling passive caucasian instagram followers,...","selling passive caucasian instagram followers,...",0,0


In [6]:
FEATURE_ARTIFACT_PATH = os.path.join(CHECKPOINT_DIR, "feature_artifacts.joblib")

X_TRAIN_PATH = os.path.join(CHECKPOINT_DIR, "X_train_features.npz")
X_VAL_PATH = os.path.join(CHECKPOINT_DIR, "X_val_features.npz")
X_TEST_PATH = os.path.join(CHECKPOINT_DIR, "X_test_features.npz")

Y_PATH = os.path.join(CHECKPOINT_DIR, "labels.joblib")

def build_tfidf_union():
    return FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            analyzer="word",
            ngram_range=WORD_NGRAM_RANGE,
            min_df=WORD_MIN_DF,
            max_df=0.90,
            max_features=WORD_MAX_FEATURES,
            sublinear_tf=True,
            dtype=np.float32
        )),
        ("char_tfidf", TfidfVectorizer(
            analyzer=CHAR_ANALYZER,
            ngram_range=CHAR_NGRAM_RANGE,
            min_df=CHAR_MIN_DF,
            max_features=CHAR_MAX_FEATURES,
            sublinear_tf=True,
            dtype=np.float32
        ))
    ])

def build_feature_matrices(train_df, val_df, test_df, force_rebuild=False):
    cache_exists = (
        os.path.exists(FEATURE_ARTIFACT_PATH)
        and os.path.exists(X_TRAIN_PATH)
        and os.path.exists(X_VAL_PATH)
        and os.path.exists(X_TEST_PATH)
        and os.path.exists(Y_PATH)
    )

    if cache_exists and not force_rebuild:
        print("Loading cached feature matrices...")
        artifacts = joblib.load(FEATURE_ARTIFACT_PATH)
        X_train = load_npz(X_TRAIN_PATH)
        X_val = load_npz(X_VAL_PATH)
        X_test = load_npz(X_TEST_PATH)
        labels = joblib.load(Y_PATH)
        return X_train, X_val, X_test, labels, artifacts

    print("Building feature matrices from scratch...")

    tfidf_union = build_tfidf_union()

    t0 = time.perf_counter()
    X_train_tfidf = tfidf_union.fit_transform(train_df["cleaned_text"])
    X_val_tfidf = tfidf_union.transform(val_df["cleaned_text"])
    X_test_tfidf = tfidf_union.transform(test_df["cleaned_text"])
    tfidf_time = time.perf_counter() - t0

    print("TF-IDF shapes:")
    print("  train:", X_train_tfidf.shape)
    print("  val  :", X_val_tfidf.shape)
    print("  test :", X_test_tfidf.shape)
    print(f"TF-IDF time: {tfidf_time:.2f}s")

    k_eff = min(FEATURE_K, X_train_tfidf.shape[1])
    print(f"Applying Chi-square feature selection: k={k_eff}")

    t1 = time.perf_counter()
    selector = SelectKBest(score_func=chi2, k=k_eff)
    X_train_selected = selector.fit_transform(X_train_tfidf, train_df["status"])
    X_val_selected = selector.transform(X_val_tfidf)
    X_test_selected = selector.transform(X_test_tfidf)
    chi2_time = time.perf_counter() - t1
    print(f"Chi-square time: {chi2_time:.2f}s")

    scaler = MinMaxScaler()
    X_train_lex = scaler.fit_transform(train_df[LEXICON_FEATURES].values)
    X_val_lex = scaler.transform(val_df[LEXICON_FEATURES].values)
    X_test_lex = scaler.transform(test_df[LEXICON_FEATURES].values)

    X_train = hstack([X_train_selected, csr_matrix(X_train_lex)], format="csr")
    X_val = hstack([X_val_selected, csr_matrix(X_val_lex)], format="csr")
    X_test = hstack([X_test_selected, csr_matrix(X_test_lex)], format="csr")

    print("Final feature shapes:")
    print("  train:", X_train.shape)
    print("  val  :", X_val.shape)
    print("  test :", X_test.shape)

    labels = {
        "y_train": train_df["status"].values,
        "y_val": val_df["status"].values,
        "y_test": test_df["status"].values,
        "class_list": sorted(train_df["status"].unique().tolist())
    }

    artifacts = {
        "tfidf_union": tfidf_union,
        "chi2_selector": selector,
        "lexicon_scaler": scaler,
        "lexicon_features": LEXICON_FEATURES,
        "feature_config": {
            "WORD_NGRAM_RANGE": WORD_NGRAM_RANGE,
            "CHAR_NGRAM_RANGE": CHAR_NGRAM_RANGE,
            "CHAR_ANALYZER": CHAR_ANALYZER,
            "WORD_MAX_FEATURES": WORD_MAX_FEATURES,
            "CHAR_MAX_FEATURES": CHAR_MAX_FEATURES,
            "WORD_MIN_DF": WORD_MIN_DF,
            "CHAR_MIN_DF": CHAR_MIN_DF,
            "FEATURE_K": FEATURE_K,
            "k_eff": k_eff
        },
        "timing": {
            "tfidf_time_seconds": tfidf_time,
            "chi2_time_seconds": chi2_time
        }
    }

    save_npz(X_TRAIN_PATH, X_train)
    save_npz(X_VAL_PATH, X_val)
    save_npz(X_TEST_PATH, X_test)
    joblib.dump(labels, Y_PATH)
    joblib.dump(artifacts, FEATURE_ARTIFACT_PATH)

    print("Feature cache saved.")
    return X_train, X_val, X_test, labels, artifacts

X_train, X_val, X_test, labels, feature_artifacts = build_feature_matrices(
    train_enriched,
    val_enriched,
    test_enriched,
    force_rebuild=FORCE_REBUILD_FEATURES
)

y_train = labels["y_train"]
y_val = labels["y_val"]
y_test = labels["y_test"]
class_list = labels["class_list"]

Loading cached feature matrices...


In [7]:
models = {
    "LinearSVC_default": LinearSVC(),
    "LogisticRegression_default": LogisticRegression(),
    "RidgeClassifier_default": RidgeClassifier(),
    "ComplementNB_default": ComplementNB(),
    "LightGBM_default": LGBMClassifier(),
    "DecisionTree_default": DecisionTreeClassifier(),
}

models


{'LinearSVC_default': LinearSVC(),
 'LogisticRegression_default': LogisticRegression(),
 'RidgeClassifier_default': RidgeClassifier(),
 'ComplementNB_default': ComplementNB(),
 'LightGBM_default': LGBMClassifier(),
 'DecisionTree_default': DecisionTreeClassifier()}

In [8]:
from IPython.display import display


def plot_confusion_matrix(cm, labels, title, save_path):
    fig, ax = plt.subplots(figsize=(7, 6))
    im = ax.imshow(cm)
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

    ax.set_xticks(np.arange(len(labels)))
    ax.set_yticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticklabels(labels)

    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, cm[i, j], ha="center", va="center")

    fig.colorbar(im, ax=ax)
    plt.tight_layout()
    plt.savefig(save_path, dpi=150)
    plt.close(fig)


def evaluate_predictions(y_true, y_pred, labels):
    report_dict = classification_report(
        y_true, y_pred, labels=labels, output_dict=True, zero_division=0
    )
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_precision": report_dict["macro avg"]["precision"],
        "macro_recall": report_dict["macro avg"]["recall"],
        "macro_f1": report_dict["macro avg"]["f1-score"],
        "weighted_f1": report_dict["weighted avg"]["f1-score"],
        "report_dict": report_dict,
        "report_text": classification_report(
            y_true, y_pred, labels=labels, zero_division=0
        ),
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=labels)
    }


def transform_enriched_dataframe(df_input: pd.DataFrame, artifacts: dict):
    """Transform an enriched dataframe into the final sparse feature matrix."""
    tfidf_union = artifacts["tfidf_union"]
    selector = artifacts["chi2_selector"]
    scaler = artifacts["lexicon_scaler"]
    lexicon_features = artifacts["lexicon_features"]

    missing_cols = ["cleaned_text"] + list(lexicon_features)
    missing_cols = [col for col in missing_cols if col not in df_input.columns]
    if missing_cols:
        raise ValueError(
            "df_input is not enriched. Missing columns: " + ", ".join(missing_cols)
        )

    X_tfidf = tfidf_union.transform(df_input["cleaned_text"].astype(str))
    X_selected = selector.transform(X_tfidf)

    X_lex = scaler.transform(df_input[lexicon_features].values)
    X_lex = csr_matrix(X_lex.astype(np.float32))

    X_final = hstack([X_selected, X_lex], format="csr")
    return X_final


def predict_enriched_pipeline_with_timing(model, df_enriched: pd.DataFrame, artifacts: dict):
    """Measure inference from enriched dataframe to final prediction."""
    start_time = time.perf_counter()
    X_input = transform_enriched_dataframe(df_enriched, artifacts)
    preds = model.predict(X_input)
    elapsed_time = time.perf_counter() - start_time
    return preds, elapsed_time


def predict_raw_text_pipeline_with_timing(model, df_raw: pd.DataFrame, artifacts: dict):
    """Measure inference from raw text dataframe to final prediction."""
    start_time = time.perf_counter()
    df_enriched = extract_lexicon_features(df_raw)
    X_input = transform_enriched_dataframe(df_enriched, artifacts)
    preds = model.predict(X_input)
    elapsed_time = time.perf_counter() - start_time
    return preds, elapsed_time


def measure_model_only_predict_time(model, X_input):
    """Measure only model.predict on an already-built feature matrix."""
    start_time = time.perf_counter()
    preds = model.predict(X_input)
    elapsed_time = time.perf_counter() - start_time
    return preds, elapsed_time


def train_or_load_model(model_name, model, X_train, y_train):
    model_path = os.path.join(CHECKPOINT_DIR, f"{model_name}.joblib")
    timing_path = os.path.join(CHECKPOINT_DIR, f"{model_name}_timing.json")

    if os.path.exists(model_path) and not FORCE_RETRAIN_MODELS:
        print(f"Loading existing model: {model_name}")
        fitted_model = joblib.load(model_path)
        timing = {}
        if os.path.exists(timing_path):
            with open(timing_path, "r", encoding="utf-8") as f:
                timing = json.load(f)
        timing.setdefault("train_time_seconds", None)
        return fitted_model, timing

    print(f"Training model: {model_name}")
    fitted_model = clone(model)

    t0 = time.perf_counter()
    fitted_model.fit(X_train, y_train)
    train_time = time.perf_counter() - t0

    joblib.dump(fitted_model, model_path)

    timing = {
        "model_name": model_name,
        "train_time_seconds": train_time,
        "trained_at": datetime.now().isoformat()
    }

    with open(timing_path, "w", encoding="utf-8") as f:
        json.dump(timing, f, indent=2, ensure_ascii=False)

    print(f"  Train time: {train_time:.2f}s")
    return fitted_model, timing


def clean_model_name(model_name: str) -> str:
    name_map = {
        "LinearSVC_default": "LinearSVC",
        "LogisticRegression_default": "Logistic Regression",
        "RidgeClassifier_default": "RidgeClassifier",
        "ComplementNB_default": "ComplementNB",
        "LightGBM_default": "LightGBM",
        "DecisionTree_default": "Decision Tree",
    }
    return name_map.get(model_name, model_name.replace("_default", ""))


def build_slide_summary_table(summary_df: pd.DataFrame) -> pd.DataFrame:
    """Create a compact table for slides using test-set metrics."""
    table = summary_df.copy()
    table["Phương pháp / Mô hình"] = table["model_name"].apply(clean_model_name)

    slide_table = table[[
        "Phương pháp / Mô hình",
        "test_accuracy",
        "test_macro_precision",
        "test_macro_recall",
        "test_macro_f1",
        "test_raw_text_pipeline_inference_time_seconds",
        "test_raw_text_pipeline_inference_time_per_sample",
    ]].rename(columns={
        "test_accuracy": "Accuracy",
        "test_macro_precision": "Precision (macro)",
        "test_macro_recall": "Recall (macro)",
        "test_macro_f1": "F1 macro",
        "test_raw_text_pipeline_inference_time_seconds": "Raw pipeline inference time (s)",
        "test_raw_text_pipeline_inference_time_per_sample": "Raw pipeline time / sample (s)",
    })

    return slide_table


def style_slide_table(slide_table: pd.DataFrame):
    return (
        slide_table.style
        .format({
            "Accuracy": "{:.4f}",
            "Precision (macro)": "{:.4f}",
            "Recall (macro)": "{:.4f}",
            "F1 macro": "{:.4f}",
            "Raw pipeline inference time (s)": "{:.4f}",
            "Raw pipeline time / sample (s)": "{:.8f}",
        })
        .set_properties(**{
            "text-align": "center",
            "border": "1px solid #333333",
            "font-size": "13px",
            "padding": "8px",
        })
        .set_table_styles([
            {
                "selector": "th",
                "props": [
                    ("background-color", "#eeeeee"),
                    ("font-weight", "bold"),
                    ("text-align", "center"),
                    ("border", "1px solid #333333"),
                    ("padding", "10px"),
                ],
            },
            {
                "selector": "td:first-child",
                "props": [("font-weight", "bold")],
            },
            {
                "selector": "table",
                "props": [
                    ("border-collapse", "collapse"),
                    ("border", "2px solid #7F3FFF"),
                ],
            },
        ])
        .hide(axis="index")
    )


def save_slide_table_png(slide_table: pd.DataFrame, save_path: str):
    """Save a simple PNG version of the summary table for direct use in slides."""
    display_table = slide_table.copy()
    numeric_cols = [
        "Accuracy",
        "Precision (macro)",
        "Recall (macro)",
        "F1 macro",
        "Raw pipeline inference time (s)",
        "Raw pipeline time / sample (s)",
    ]
    for col in numeric_cols:
        if col == "Raw pipeline time / sample (s)":
            display_table[col] = display_table[col].map(lambda x: f"{x:.8f}")
        else:
            display_table[col] = display_table[col].map(lambda x: f"{x:.4f}")

    n_rows, n_cols = display_table.shape
    fig_width = 16
    fig_height = max(2.2, 0.55 * (n_rows + 1))
    fig, ax = plt.subplots(figsize=(fig_width, fig_height))
    ax.axis("off")

    table = ax.table(
        cellText=display_table.values,
        colLabels=display_table.columns,
        loc="center",
        cellLoc="center",
        colLoc="center",
    )

    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.6)

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor("#333333")
        cell.set_linewidth(0.8)
        if row == 0:
            cell.set_facecolor("#eeeeee")
            cell.set_text_props(weight="bold")
        if row > 0 and col == 0:
            cell.set_text_props(weight="bold")

    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)


all_summary = []
all_reports = {}

for model_name, model in models.items():
    print("\n" + "=" * 80)
    print(model_name)
    print("=" * 80)

    fitted_model, timing = train_or_load_model(model_name, model, X_train, y_train)

    warmup_n = min(10, len(test_enriched))
    if warmup_n > 0:
        _ = fitted_model.predict(X_test[:warmup_n])
        _ = predict_enriched_pipeline_with_timing(
            fitted_model,
            test_enriched.iloc[:warmup_n].copy(),
            feature_artifacts
        )
        _ = predict_raw_text_pipeline_with_timing(
            fitted_model,
            test_df.iloc[:warmup_n].copy(),
            feature_artifacts
        )

    y_val_pred_model_only, val_model_only_time = measure_model_only_predict_time(
        fitted_model,
        X_val
    )
    y_val_pred_enriched, val_enriched_pipeline_time = predict_enriched_pipeline_with_timing(
        fitted_model,
        val_enriched.copy(),
        feature_artifacts
    )

    y_test_pred_model_only, test_model_only_time = measure_model_only_predict_time(
        fitted_model,
        X_test
    )
    y_test_pred_enriched, test_enriched_pipeline_time = predict_enriched_pipeline_with_timing(
        fitted_model,
        test_enriched.copy(),
        feature_artifacts
    )
    y_test_pred_raw, test_raw_text_pipeline_time = predict_raw_text_pipeline_with_timing(
        fitted_model,
        test_df.copy(),
        feature_artifacts
    )

    if not np.array_equal(y_val_pred_model_only, y_val_pred_enriched):
        print("WARNING: Validation predictions differ between model-only and enriched-pipeline timing.")

    if not np.array_equal(y_test_pred_model_only, y_test_pred_enriched):
        print("WARNING: Test predictions differ between model-only and enriched-pipeline timing.")

    if not np.array_equal(y_test_pred_model_only, y_test_pred_raw):
        print("WARNING: Test predictions differ between model-only and raw-text-pipeline timing.")

    val_eval = evaluate_predictions(y_val, y_val_pred_enriched, class_list)
    test_eval = evaluate_predictions(y_test, y_test_pred_raw, class_list)

    print("\nValidation report:")
    print(val_eval["report_text"])

    print("\nTest report:")
    print(test_eval["report_text"])

    print("\nInference timing:")
    print(f"  Test model-only time              : {test_model_only_time:.6f}s")
    print(f"  Test enriched-pipeline time       : {test_enriched_pipeline_time:.6f}s")
    print(f"  Test raw-text-pipeline time       : {test_raw_text_pipeline_time:.6f}s")
    print(f"  Test raw-text-pipeline / sample   : {test_raw_text_pipeline_time / len(y_test):.8f}s/sample")

    val_cm_path = os.path.join(FIGURE_DIR, f"{model_name}_val_confusion_matrix.png")
    test_cm_path = os.path.join(FIGURE_DIR, f"{model_name}_test_confusion_matrix.png")

    plot_confusion_matrix(
        val_eval["confusion_matrix"],
        class_list,
        f"{model_name} - Validation Confusion Matrix",
        val_cm_path
    )

    plot_confusion_matrix(
        test_eval["confusion_matrix"],
        class_list,
        f"{model_name} - Test Confusion Matrix",
        test_cm_path
    )

    model_record = {
        "model_name": model_name,
        "display_name": clean_model_name(model_name),
        "train_time_seconds": timing.get("train_time_seconds"),

        "val_inference_time_seconds": val_enriched_pipeline_time,
        "val_inference_time_per_sample": val_enriched_pipeline_time / len(y_val),
        "test_inference_time_seconds": test_raw_text_pipeline_time,
        "test_inference_time_per_sample": test_raw_text_pipeline_time / len(y_test),
        "inference_time_measurement_used": "raw_text_pipeline",

        "val_model_only_inference_time_seconds": val_model_only_time,
        "val_model_only_inference_time_per_sample": val_model_only_time / len(y_val),
        "val_enriched_pipeline_inference_time_seconds": val_enriched_pipeline_time,
        "val_enriched_pipeline_inference_time_per_sample": val_enriched_pipeline_time / len(y_val),
        "test_model_only_inference_time_seconds": test_model_only_time,
        "test_model_only_inference_time_per_sample": test_model_only_time / len(y_test),
        "test_enriched_pipeline_inference_time_seconds": test_enriched_pipeline_time,
        "test_enriched_pipeline_inference_time_per_sample": test_enriched_pipeline_time / len(y_test),
        "test_raw_text_pipeline_inference_time_seconds": test_raw_text_pipeline_time,
        "test_raw_text_pipeline_inference_time_per_sample": test_raw_text_pipeline_time / len(y_test),

        "val_accuracy": val_eval["accuracy"],
        "val_macro_precision": val_eval["macro_precision"],
        "val_macro_recall": val_eval["macro_recall"],
        "val_macro_f1": val_eval["macro_f1"],
        "val_weighted_f1": val_eval["weighted_f1"],
        "test_accuracy": test_eval["accuracy"],
        "test_macro_precision": test_eval["macro_precision"],
        "test_macro_recall": test_eval["macro_recall"],
        "test_macro_f1": test_eval["macro_f1"],
        "test_weighted_f1": test_eval["weighted_f1"],
        "val_confusion_matrix_path": val_cm_path,
        "test_confusion_matrix_path": test_cm_path
    }

    all_summary.append(model_record)

    all_reports[model_name] = {
        "validation": {
            "metrics": {
                "accuracy": val_eval["accuracy"],
                "macro_precision": val_eval["macro_precision"],
                "macro_recall": val_eval["macro_recall"],
                "macro_f1": val_eval["macro_f1"],
                "weighted_f1": val_eval["weighted_f1"]
            },
            "classification_report": val_eval["report_dict"],
            "confusion_matrix": val_eval["confusion_matrix"].tolist()
        },
        "test": {
            "metrics": {
                "accuracy": test_eval["accuracy"],
                "macro_precision": test_eval["macro_precision"],
                "macro_recall": test_eval["macro_recall"],
                "macro_f1": test_eval["macro_f1"],
                "weighted_f1": test_eval["weighted_f1"]
            },
            "classification_report": test_eval["report_dict"],
            "confusion_matrix": test_eval["confusion_matrix"].tolist()
        },
        "timing": {
            "measurement_note": (
                "model_only measures model.predict on precomputed X; "
                "enriched_pipeline starts from cleaned_text + lexicon columns; "
                "raw_text_pipeline starts from raw text and includes lexicon feature creation."
            ),
            "train_time_seconds": timing.get("train_time_seconds"),
            "val_model_only_inference_time_seconds": val_model_only_time,
            "val_model_only_inference_time_per_sample": val_model_only_time / len(y_val),
            "val_enriched_pipeline_inference_time_seconds": val_enriched_pipeline_time,
            "val_enriched_pipeline_inference_time_per_sample": val_enriched_pipeline_time / len(y_val),
            "test_model_only_inference_time_seconds": test_model_only_time,
            "test_model_only_inference_time_per_sample": test_model_only_time / len(y_test),
            "test_enriched_pipeline_inference_time_seconds": test_enriched_pipeline_time,
            "test_enriched_pipeline_inference_time_per_sample": test_enriched_pipeline_time / len(y_test),
            "test_raw_text_pipeline_inference_time_seconds": test_raw_text_pipeline_time,
            "test_raw_text_pipeline_inference_time_per_sample": test_raw_text_pipeline_time / len(y_test)
        }
    }

summary_df = pd.DataFrame(all_summary).sort_values(
    by="test_macro_f1",
    ascending=False
).reset_index(drop=True)

slide_summary_df = build_slide_summary_table(summary_df)

summary_path = os.path.join(REPORT_DIR, "default_models_summary.csv")
slide_summary_path = os.path.join(REPORT_DIR, "default_models_slide_summary.csv")
slide_html_path = os.path.join(REPORT_DIR, "default_models_slide_summary.html")
slide_png_path = os.path.join(FIGURE_DIR, "default_models_slide_summary.png")
reports_path = os.path.join(REPORT_DIR, "default_models_reports.json")

summary_df.to_csv(summary_path, index=False)
slide_summary_df.to_csv(slide_summary_path, index=False)

styled_table = style_slide_table(slide_summary_df)
styled_table.to_html(slide_html_path)
save_slide_table_png(slide_summary_df, slide_png_path)

with open(reports_path, "w", encoding="utf-8") as f:
    json.dump(all_reports, f, indent=2, ensure_ascii=False)

print("\nSaved full summary to:", summary_path)
print("Saved slide summary CSV to:", slide_summary_path)
print("Saved slide summary HTML to:", slide_html_path)
print("Saved slide summary PNG to:", slide_png_path)
print("Saved reports to:", reports_path)

print("\nFull summary:")
display(summary_df)

print("\nSlide-style summary table:")
display(styled_table)




LinearSVC_default
Loading existing model: LinearSVC_default

Validation report:
              precision    recall  f1-score   support

     anxiety       0.84      0.79      0.82       557
  depression       0.73      0.70      0.71      1451
      normal       0.89      0.95      0.92      1812
    suicidal       0.69      0.66      0.68      1119

    accuracy                           0.79      4939
   macro avg       0.79      0.78      0.78      4939
weighted avg       0.79      0.79      0.79      4939


Test report:
              precision    recall  f1-score   support

     anxiety       0.84      0.79      0.81      1114
  depression       0.72      0.69      0.70      2900
      normal       0.89      0.94      0.92      3625
    suicidal       0.68      0.66      0.67      2236

    accuracy                           0.79      9875
   macro avg       0.78      0.77      0.78      9875
weighted avg       0.79      0.79      0.79      9875


Inference timing:
  Test model-onl

,model_name,display_name,train_time_seconds,val_inference_time_seconds,val_inference_time_per_sample,test_inference_time_seconds,test_inference_time_per_sample,inference_time_measurement_used,val_model_only_inference_time_seconds,val_model_only_inference_time_per_sample,...,val_macro_recall,val_macro_f1,val_weighted_f1,test_accuracy,test_macro_precision,test_macro_recall,test_macro_f1,test_weighted_f1,val_confusion_matrix_path,test_confusion_matrix_path
0,LightGBM_default,LightGBM,781.154664,2.669263,0.000540,8.010317,0.000811,raw_text_pipeline,0.228889,0.000046,...,0.792959,0.800046,0.808227,0.814380,0.809683,0.797990,0.803415,0.813244,/content/drive/MyDrive/CS114/TeamModel/experim...,/content/drive/MyDrive/CS114/TeamModel/experim...
1,LogisticRegression_default,Logistic Regression,37.669048,2.281062,0.000462,7.507758,0.000760,raw_text_pipeline,0.011497,0.000002,...,0.782090,0.789194,0.800319,0.802532,0.796851,0.782183,0.788701,0.800242,/content/drive/MyDrive/CS114/TeamModel/experim...,/content/drive/MyDrive/CS114/TeamModel/experim...
2,LinearSVC_default,LinearSVC,10.282785,2.830913,0.000573,8.253274,0.000836,raw_text_pipeline,0.016902,0.000003,...,0.776652,0.781065,0.791717,0.789165,0.780717,0.772826,0.776404,0.787210,/content/drive/MyDrive/CS114/TeamModel/experim...,/content/drive/MyDrive/CS114/TeamModel/experim...
3,RidgeClassifier_default,RidgeClassifier,25.290002,2.257758,0.000457,8.461143,0.000857,raw_text_pipeline,0.012837,0.000003,...,0.767513,0.777623,0.788448,0.785114,0.780541,0.762773,0.770342,0.781920,/content/drive/MyDrive/CS114/TeamModel/experim...,/content/drive/MyDrive/CS114/TeamModel/experim...
4,ComplementNB_default,ComplementNB,0.232846,2.711322,0.000549,9.443776,0.000956,raw_text_pipeline,0.013301,0.000003,...,0.747786,0.729872,0.742726,0.743190,0.721864,0.747331,0.729646,0.742114,/content/drive/MyDrive/CS114/TeamModel/experim...,/content/drive/MyDrive/CS114/TeamModel/experim...
5,DecisionTree_default,Decision Tree,318.479999,2.270159,0.000460,8.350869,0.000846,raw_text_pipeline,0.010508,0.000002,...,0.653174,0.656661,0.677658,0.690025,0.667326,0.669100,0.668127,0.689613,/content/drive/MyDrive/CS114/TeamModel/experim...,/content/drive/MyDrive/CS114/TeamModel/experim...



Slide-style summary table:


Phương pháp / Mô hình,Accuracy,Precision (macro),Recall (macro),F1 macro,Raw pipeline inference time (s),Raw pipeline time / sample (s)
LightGBM,0.8144,0.8097,0.7980,0.8034,8.0103,0.00081117
Logistic Regression,0.8025,0.7969,0.7822,0.7887,7.5078,0.00076028
LinearSVC,0.7892,0.7807,0.7728,0.7764,8.2533,0.00083577
RidgeClassifier,0.7851,0.7805,0.7628,0.7703,8.4611,0.00085682
ComplementNB,0.7432,0.7219,0.7473,0.7296,9.4438,0.00095633
Decision Tree,0.6900,0.6673,0.6691,0.6681,8.3509,0.00084566


In [9]:

config = {
    "experiment_name": EXPERIMENT_NAME,
    "created_at": datetime.now().isoformat(),
    "feature_design": {
        "description": (
            "Word TF-IDF + character TF-IDF + targeted lexicon-based features. "
            "Chi-square is applied only to TF-IDF features. Lexicon features are appended after Chi-square."
        ),
        "lexicon_features": LEXICON_FEATURES,
        "depression_keywords": DEPRESSION_KEYWORDS,
        "suicide_keywords": SUICIDE_KEYWORDS,
        "feature_config": feature_artifacts.get("feature_config", {})
    },
    "models": {
        name: repr(model) for name, model in models.items()
    },
    "notes": [
        "All models use default hyperparameters.",
        "RidgeClassifier is used as the classification equivalent of Ridge Regression.",
        "No surface-level handcrafted features are used.",
        "Only targeted lexicon-based numeric features are included."
    ],
    "paths": {
        "experiment_dir": EXP_DIR,
        "checkpoint_dir": CHECKPOINT_DIR,
        "report_dir": REPORT_DIR,
        "figure_dir": FIGURE_DIR
    }
}

config_path = os.path.join(REPORT_DIR, "experiment_config.json")

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2, ensure_ascii=False)

print("Saved config:", config_path)

Saved config: /content/drive/MyDrive/CS114/TeamModel/experiments/tfidf_lexicon_only_all_default_models_decision_tree_v1/reports/experiment_config.json


In [10]:
val_result_table = summary_df[[
    "display_name",
    "val_accuracy",
    "val_macro_precision",
    "val_macro_recall",
    "val_macro_f1",
    "val_weighted_f1"
]].sort_values(
    by="val_macro_f1",
    ascending=False
).reset_index(drop=True)

val_result_table

,display_name,val_accuracy,val_macro_precision,val_macro_recall,val_macro_f1,val_weighted_f1
0,LightGBM,0.810488,0.808987,0.792959,0.800046,0.808227
1,Logistic Regression,0.803199,0.798501,0.782090,0.789194,0.800319
2,LinearSVC,0.794088,0.786618,0.776652,0.781065,0.791717
3,RidgeClassifier,0.792063,0.791825,0.767513,0.777623,0.788448
4,ComplementNB,0.744280,0.722474,0.747786,0.729872,0.742726
5,Decision Tree,0.679085,0.660727,0.653174,0.656661,0.677658
